<a href="https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HamzaMohamed25/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis (the grain).** One row is one content item for one client: the pair `client_hash_id` + `content_hash_id`. The warehouse stores one row per page per day, so each month is aggregated into one row per page.

**Time window (a decision point).** The decision is made on **1 May 2026**. Features come from **February, March and April 2026**. The label comes from **May 2026**. **June 2026 stays sealed** and is never loaded. Each month is read from its own partition (`month=YYYY-MM`) and the code checks that every date falls inside its month.

**Who is eligible.** A page needs at least 1,000 impressions across Feb-Apr and at least 10 clicks in April. That leaves **16,513 of 380,147 pages from 36 clients**. These thresholds were fixed before any model.

**Why this replaced my first draft.** My first draft used March as the window and a same-month label (impressions below the median). That label was a function of one of the input features, so any model would have scored perfectly without learning anything. The decision-point design fixes it: inputs end before the label window starts.


In [1]:
import gc, json, os
import numpy as np, pandas as pd

def hf_token():
    try:
        from google.colab import userdata          # Colab: Secrets panel
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")          # local run: environment variable

HF = {"token": hf_token()}
HF_BASE = "hf://datasets/FlyRank/internship-warehouse"
FEAT_MONTHS, LABEL_MONTH = ["2026-02", "2026-03", "2026-04"], "2026-05"
MIN_IMPR_WINDOW, MIN_CLICKS_APR, DROP_RATIO = 1000, 10, 0.80      # contract thresholds: frozen before any model
KEYS = ["client_hash_id", "content_hash_id"]
COLS = ["report_date", *KEYS, "gsc_clicks", "gsc_impressions", "gsc_avg_position"]
assert LABEL_MONTH != "2026-06" and "2026-06" not in FEAT_MONTHS, "June is the sealed test month"

def month_agg(m):
    """One partition -> one row per (client, content). Loads only needed columns; frees RAM right after."""
    path = f"{HF_BASE}/fact_content_daily_performance/month={m}/"
    try:
        d = pd.read_parquet(path, columns=COLS, storage_options=HF)
    except Exception:
        from huggingface_hub import HfFileSystem
        import pyarrow.parquet as pq
        fs = HfFileSystem(token=HF["token"])
        f = fs.ls(f"datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={m}", detail=False)[0]
        print("Actual schema:", pq.read_schema(f, filesystem=fs).names); raise
    dates = pd.to_datetime(d["report_date"])
    win = (dates.min().date(), dates.max().date(), len(d))
    impr = d["gsc_impressions"].fillna(0)
    pos = d["gsc_avg_position"].where(d["gsc_avg_position"] > 0)            # 0/negative = no data, not rank 0
    d = d.assign(clicks=d["gsc_clicks"].fillna(0), impr=impr,
                 pos_x_impr=(pos * impr).fillna(0), impr_pos=impr.where(pos.notna(), 0),
                 active=(impr > 0).astype("int16"))
    g = d.groupby(KEYS, observed=True).agg(clicks=("clicks", "sum"), impr=("impr", "sum"),
                                           pos_x_impr=("pos_x_impr", "sum"), impr_pos=("impr_pos", "sum"),
                                           active_days=("active", "sum")).reset_index()
    del d; gc.collect()
    return g, win

# 1) Aggregate each month separately (daily grain -> content grain), check windows land inside their month
frames, windows = {}, {}
for m in FEAT_MONTHS + [LABEL_MONTH]:
    frames[m], windows[m] = month_agg(m)
    assert str(windows[m][0]).startswith(m) and str(windows[m][1]).startswith(m), f"{m} partition has dates outside its month"
    print(f"{m}: daily rows={windows[m][2]:,} | {windows[m][0]} -> {windows[m][1]} | contents={len(frames[m]):,}")

# 2) Feature frame from Feb-Apr ONLY (nine features, all knowable on 2026-05-01)
f = None
for m in FEAT_MONTHS:
    part = frames[m].rename(columns={c: f"{c}_{m}" for c in ["clicks", "impr", "pos_x_impr", "impr_pos", "active_days"]})
    f = part if f is None else f.merge(part, on=KEYS, how="outer")
num = [c for c in f.columns if c not in KEYS]
f[num] = f[num].fillna(0)                                             # absent row in a month = no search rows that month
S = lambda p: sum(f[f"{p}_{m}"] for m in FEAT_MONTHS)
f["impressions_total"], f["clicks_total"] = S("impr"), S("clicks")
f["impressions_apr"], f["clicks_apr"], f["clicks_feb"] = f["impr_2026-04"], f["clicks_2026-04"], f["clicks_2026-02"]
f["momentum_apr_vs_feb"] = np.log1p(f["clicks_apr"]) - np.log1p(f["clicks_feb"])
f["ctr"] = np.where(f["impressions_total"] > 0, f["clicks_total"] / f["impressions_total"], np.nan)
f["weighted_position"] = np.where(S("impr_pos") > 0, S("pos_x_impr") / S("impr_pos").replace(0, np.nan), np.nan)  # sum(pos*impr)/sum(impr), not a mean of daily means
f["active_days"] = S("active_days")
FEATURES = ["impressions_total", "clicks_total", "impressions_apr", "clicks_apr", "clicks_feb",
            "momentum_apr_vs_feb", "ctr", "weighted_position", "active_days"]
f = f[KEYS + FEATURES]

# 3) Label from May ONLY: decline = May clicks < 80% of April clicks. Label ingredients are never features.
may = frames[LABEL_MONTH][KEYS + ["clicks"]].rename(columns={"clicks": "clicks_may"})
n_before_eligibility = len(f)
f = f[(f["impressions_total"] >= MIN_IMPR_WINDOW) & (f["clicks_apr"] >= MIN_CLICKS_APR)]   # eligibility, decided at the decision point
f = f.merge(may, on=KEYS, how="left")
missing_in_may = int(f["clicks_may"].isna().sum())
f["clicks_may"] = f["clicks_may"].fillna(0)                           # not seen in May -> 0 clicks -> counts as decline (stated in contract)
f["y_decline_may"] = (f["clicks_may"] < DROP_RATIO * f["clicks_apr"]).astype(int)
LABEL_INGREDIENTS = ["clicks_may"]                                    # kept ONLY to build/audit the label
frame = f.reset_index(drop=True)

# 4) Contract checks (queries, not claims)
assert not frame.duplicated(KEYS).any(), "grain broken: (client, content) must be unique"
assert not set(LABEL_INGREDIENTS) & set(FEATURES), "label ingredient leaked into features"
assert set(KEYS).isdisjoint(FEATURES), "IDs are grouping keys, never features"
share = frame["client_hash_id"].value_counts(normalize=True)
base_rate = float(frame["y_decline_may"].mean())
print(f"\nBefore eligibility: {n_before_eligibility:,} | Eligible rows: {len(frame):,} | Clients: {frame['client_hash_id'].nunique()}")
print(f"Base rate (decline in May): {base_rate:.3f} | Not seen in May (counted as decline): {missing_in_may:,}")
print(f"Largest client share of rows: {share.iloc[0]:.1%} | Top-5 clients: {share.head(5).sum():.1%}")
print("Per-client base rate spread (min / median / max, clients with >=30 rows):")
pc = frame.groupby("client_hash_id")["y_decline_may"].agg(["mean", "size"]); pc = pc[pc["size"] >= 30]["mean"]
print(f"  {pc.min():.2f} / {pc.median():.2f} / {pc.max():.2f}  over {len(pc)} clients")
print("Missing in features (share):"); print(frame[FEATURES].isna().mean().round(3).to_string())

# 5) Receipt (commit this JSON), cache the frame locally (do NOT commit the parquet)
os.makedirs("work/outputs", exist_ok=True)
frame.to_parquet("work/outputs/frame_dp_2026-05-01.parquet", index=False)
json.dump({"decision_point": "2026-05-01", "feature_months": FEAT_MONTHS, "label_month": LABEL_MONTH,
           "min_impr_window": MIN_IMPR_WINDOW, "min_clicks_apr": MIN_CLICKS_APR, "drop_ratio": DROP_RATIO,
           "rows_before_eligibility": int(n_before_eligibility), "rows": int(len(frame)),
           "clients": int(frame["client_hash_id"].nunique()), "base_rate": round(base_rate, 4),
           "not_seen_in_may": missing_in_may, "features": FEATURES, "label": "y_decline_may",
           "group_key": "client_hash_id", "sealed_month": "2026-06", "release": "v20260703"},
          open("work/outputs/data_contract_v2.json", "w"), indent=2)
print("\nSaved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)")

2026-02: daily rows=7,355,108 | 2026-02-01 -> 2026-02-28 | contents=321,546
2026-03: daily rows=9,841,378 | 2026-03-01 -> 2026-03-31 | contents=331,437
2026-04: daily rows=10,424,730 | 2026-04-01 -> 2026-04-30 | contents=362,172
2026-05: daily rows=11,687,376 | 2026-05-01 -> 2026-05-31 | contents=389,153

Before eligibility: 380,147 | Eligible rows: 16,513 | Clients: 36
Base rate (decline in May): 0.416 | Not seen in May (counted as decline): 0
Largest client share of rows: 26.1% | Top-5 clients: 71.9%
Per-client base rate spread (min / median / max, clients with >=30 rows):
  0.03 / 0.38 / 0.75  over 17 clients
Missing in features (share):
impressions_total      0.0
clicks_total           0.0
impressions_apr        0.0
clicks_apr             0.0
clicks_feb             0.0
momentum_apr_vs_feb    0.0
ctr                    0.0
weighted_position      0.0
active_days            0.0

Saved: work/outputs/data_contract_v2.json (commit) + frame parquet (cache only)


## 2. Fields: feature / label / context / excluded

**Features (9, all knowable on 1 May 2026):** `impressions_total`, `clicks_total`, `impressions_apr`, `clicks_apr`, `clicks_feb`, `momentum_apr_vs_feb` (log of April clicks minus log of February clicks), `ctr`, `weighted_position` and `active_days`. Position is weighted by impressions (sum of position x impressions divided by sum of impressions), not an average of daily averages.

**Label (an observed proxy, not a business outcome):** `y_decline_may = 1` when May clicks are below **80% of April clicks**. Observed base rate: **41.6%**. A page not seen in May counts as 0 clicks (0 pages were in that case).

**Context (never features):** `client_hash_id` is the grouping key for client-grouped validation, and `content_hash_id` is for joining. Both are hashed.

**Excluded:** `clicks_may` and every other May metric (they build the label), all June 2026 data, and the session, scroll and AI-referral fields (they exist only for some pages, and a zero before tracking started does not mean nothing happened).

In [2]:
LABEL = "y_decline_may"
CONTEXT = ["client_hash_id", "content_hash_id"]
EXCLUDED = ["clicks_may (label ingredient)", "every other May metric", "all June 2026 data (sealed)", "session / scroll / AI-referral fields (sparse, tracking gaps)"]
print("Features (knowable on 2026-05-01):", FEATURES)
print("Label (observed proxy, May):", LABEL)
print("Context (grouping / join only, never a feature):", CONTEXT)
print("Excluded:", EXCLUDED)
assert set(FEATURES).isdisjoint(CONTEXT) and "clicks_may" not in FEATURES and LABEL not in FEATURES

Features (knowable on 2026-05-01): ['impressions_total', 'clicks_total', 'impressions_apr', 'clicks_apr', 'clicks_feb', 'momentum_apr_vs_feb', 'ctr', 'weighted_position', 'active_days']
Label (observed proxy, May): y_decline_may
Context (grouping / join only, never a feature): ['client_hash_id', 'content_hash_id']
Excluded: ['clicks_may (label ingredient)', 'every other May metric', 'all June 2026 data (sealed)', 'session / scroll / AI-referral fields (sparse, tracking gaps)']


## 3. Verify it with queries (grain, counts, missing values, windows)

The cell below proves the contract with queries instead of claims:

- **Grain:** no duplicate (client, content) pair.
- **Windows:** Feb 1 to Apr 30 feed the features, May feeds the label, and the date range of each month is printed.
- **Missing values:** no missing values in the 9 features.
- **The trap:** a column built from the label ingredient (`clicks_may / clicks_apr`) separates the label perfectly (AUC 1.000), while every clean feature stays clearly below that. A near-perfect score from a column that uses May is a warning, not a win, so it can never enter the features.


In [3]:
from sklearn.metrics import roc_auc_score

print(f"Rows: {len(frame):,} | unique (client, content) pairs: {len(frame.drop_duplicates(KEYS)):,} | grain holds: {not frame.duplicated(KEYS).any()}")
print("Date windows used (first day -> last day, daily rows):")
for m, (a, b, n) in windows.items():
    print(f"  {m}: {a} -> {b} | {n:,} rows | role: {'LABEL month' if m == LABEL_MONTH else 'feature month'}")
print("\nMissing values in features (count):"); print(frame[FEATURES].isna().sum().to_string())

def auc(x):                                   # direction-free single-feature AUC against the label
    a = roc_auc_score(frame[LABEL], x.fillna(x.median())); return max(a, 1 - a)
clean = {c: round(auc(frame[c]), 3) for c in FEATURES}
leak = auc(frame["clicks_may"] / frame["clicks_apr"])
print("\nTHE TRAP: single-feature AUC against the label")
print("  clean features, best one:", max(clean, key=clean.get), clean[max(clean, key=clean.get)])
print(f"  a column built from the label ingredient (clicks_may / clicks_apr): {leak:.3f}")
print("A near-perfect score from a column that uses May is a warning, not a win. That column stays out of the features.")

Rows: 16,513 | unique (client, content) pairs: 16,513 | grain holds: True
Date windows used (first day -> last day, daily rows):
  2026-02: 2026-02-01 -> 2026-02-28 | 7,355,108 rows | role: feature month
  2026-03: 2026-03-01 -> 2026-03-31 | 9,841,378 rows | role: feature month
  2026-04: 2026-04-01 -> 2026-04-30 | 10,424,730 rows | role: feature month
  2026-05: 2026-05-01 -> 2026-05-31 | 11,687,376 rows | role: LABEL month

Missing values in features (count):
impressions_total      0
clicks_total           0
impressions_apr        0
clicks_apr             0
clicks_feb             0
momentum_apr_vs_feb    0
ctr                    0
weighted_position      0
active_days            0

THE TRAP: single-feature AUC against the label
  clean features, best one: momentum_apr_vs_feb 0.549
  a column built from the label ingredient (clicks_may / clicks_apr): 1.000
A near-perfect score from a column that uses May is a warning, not a win. That column stays out of the features.


## 4. Data limits

- **One decision point.** Everything here is observed on 1 May 2026 only. A different month could look different.
- **The label is a proxy.** It measures "May clicks fell below 80% of April clicks". It does not tell us why, and it partly reflects regression to the mean: a page with an unusually high April can fall back without being in decay.
- **Client concentration.** The largest client holds 26.1% of the rows and the top five hold 71.9%. Client base rates range from 0.03 to 0.75 (clients with at least 30 rows), so a random split would let a model learn the client instead of the pattern. Validation must keep each client on one side.
- **A block of identical rows.** 833 pages from a single client have an identical Feb-Apr feature vector, and 99.6% of them declined in May. Identical rows are not independent evidence. They stay in this contract frame, and the downstream notebooks exclude them (rule: a feature vector repeated 20 or more times).
- **No intent or market context, and no causality.** Search Console cannot say why impressions or clicks moved. This work is directional decision-support, not proof that a refresh would help.

In [4]:
share = frame["client_hash_id"].value_counts(normalize=True)
pc = frame.groupby("client_hash_id")[LABEL].agg(["mean", "size"]); pc = pc[pc["size"] >= 30]["mean"]
print(f"Largest client share of rows: {share.iloc[0]:.1%} | top-5 clients: {share.head(5).sum():.1%}")
print(f"Per-client base rate (clients with >=30 rows): min {pc.min():.2f} / median {pc.median():.2f} / max {pc.max():.2f} over {len(pc)} clients")
vec = ["clicks_feb", "clicks_mar", "clicks_apr", "impressions_apr", "weighted_position", "active_days"]
d = frame.assign(clicks_mar=(frame["clicks_total"] - frame["clicks_apr"] - frame["clicks_feb"]).clip(lower=0))
rep = d[d.groupby(vec, dropna=False)["content_hash_id"].transform("size") >= 20]
if len(rep):
    print(f"Rows whose Feb-Apr vector repeats >=20 times: {len(rep):,} | clients: {rep['client_hash_id'].nunique()} | decline rate there: {rep[LABEL].mean():.3f} | elsewhere: {d.drop(rep.index)[LABEL].mean():.3f}")
else:
    print("No Feb-Apr vector repeats >=20 times.")

Largest client share of rows: 26.1% | top-5 clients: 71.9%
Per-client base rate (clients with >=30 rows): min 0.03 / median 0.38 / max 0.75 over 17 clients
Rows whose Feb-Apr vector repeats >=20 times: 833 | clients: 1 | decline rate there: 0.996 | elsewhere: 0.385


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.